# 순환 신경망 기반 감성 분석 (무신사 뷰티 리뷰)
1. 데이터 준비
2. 모델 구조 및 학습 설계
3. 모델 학습
4. 모델 평가
5. 예측
6. 배포 (저장, 재사용)

## 1. 데이터 준비
    1-1. 데이터 로딩
    1-2. 데이터 전처리
    1-3. 데이터 분리
    1-4. 학습용 데이터 준비
    1-5. 테스트용 데이터 준비

### 1-1. 데이터 로딩

In [ ]:
import pandas as pd

datafile = './musinsa_beauty_TOTAL.csv'
review_df = pd.read_csv(datafile, encoding='utf-8-sig', low_memory=False)
print(f'전체 데이터: {len(review_df):,}건')
review_df.head()

In [ ]:
# sentiment 컬럼 확인
review_df['sentiment'].value_counts()

In [ ]:
# 중립 제외 → 긍정/부정 이진 분류
# 교수님 노트북과 동일하게 label: 부정=0, 긍정=1
review_df = review_df[review_df['sentiment'] != '중립'].copy()
review_df['label'] = review_df['sentiment'].map({'긍정': 1, '부정': 0})

# review_text를 document로 통일 (교수님 코드와 호환)
review_df = review_df[['review_text', 'label']].rename(columns={'review_text': 'document'})
print(f'분류 대상: {len(review_df):,}건')
review_df['label'].value_counts()

### 1-2. 데이터 전처리
- 결측치 제거
- 정제 (한글만 남기고 모두 삭제)
- 중복치 제거
- 형태소 분석기로 토큰화

#### 1-2-1. 결측치 제거

In [ ]:
review_df.isnull().sum()

In [ ]:
review_df.dropna(inplace=True)
review_df.isnull().sum()

#### 1-2-2. 정제 (한글과 공백만 남기기)

In [ ]:
import re

review_df['clean_review'] = review_df['document'].apply(lambda x: re.sub('[^ 가-힣]+', ' ', str(x)))
review_df['clean_review'] = review_df['clean_review'].apply(lambda x: re.sub('^ +', '', x))
review_df['clean_review'] = review_df['clean_review'].replace('', None)

print(f'결측치: {review_df.clean_review.isnull().sum()}건')
review_df.dropna(subset=['clean_review'], inplace=True)
review_df.head()

#### 1-2-3. 중복치 제거

In [ ]:
print(f'중복: {review_df.clean_review.duplicated().sum()}건')
review_df.drop_duplicates(subset=['clean_review'], inplace=True)
print(f'중복 제거 후: {len(review_df):,}건')
review_df['label'].value_counts()

#### 1-2-4. 형태소 분석 (토큰화)
※ 시간이 오래 걸립니다 (10~30분)

In [ ]:
from konlpy.tag import Okt
from tqdm import tqdm
tqdm.pandas()

review_df['tokens'] = review_df['clean_review'].progress_apply(Okt().morphs)
review_df['tokens_str'] = review_df['tokens'].apply(lambda x: ' '.join(x))
review_df.head()

In [ ]:
# 전처리 결과 저장 (다음에 다시 안 해도 되도록)
review_df.to_csv('./musinsa_beauty_preprocessed.csv', encoding='utf-8-sig', index=False)
print('저장 완료')

### 1-3. 데이터 분리

In [ ]:
from sklearn.model_selection import train_test_split

review_list = list(review_df['tokens_str'])
label_list  = list(review_df['label'])

review_df['label'].value_counts().plot(kind='bar', title='긍정/부정 분포')

review_train, review_test, label_train, label_test = train_test_split(
    review_list, label_list, test_size=0.1, stratify=label_list, random_state=42
)
print(f'학습: {len(review_train):,}건 / 테스트: {len(review_test):,}건')

### 1-4. 학습 데이터 준비

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer

# 단어 수 확인용
test_tokenizer = Tokenizer()
test_tokenizer.fit_on_texts(review_train)
print(f'전체 단어 수: {len(test_tokenizer.word_index):,}개')

In [ ]:
vocab_size = 40000
num_words  = vocab_size + 1
tokenizer  = Tokenizer(num_words=num_words)
tokenizer.fit_on_texts(review_train)

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Integer Encoding
encoded_train = tokenizer.texts_to_sequences(review_train)

# 길이 0인 리뷰 제거
null_index    = [i for i, r in enumerate(encoded_train) if len(r) < 1]
new_review_train = [r for i, r in enumerate(encoded_train) if i not in null_index]
new_label_train  = [l for i, l in enumerate(label_train)  if i not in null_index]
print(f'유효 학습 데이터: {len(new_review_train):,}건')

# 리뷰 길이 분포 확인
import pandas as pd
len_df = pd.DataFrame([len(r) for r in new_review_train])
print(len_df.describe())

# Padding
max_len = 100  # 뷰티 리뷰는 영화리뷰보다 길어서 100으로 설정
train_X = pad_sequences(new_review_train, maxlen=max_len)
print(f'train_X shape: {train_X.shape}')

In [ ]:
from tensorflow.keras.utils import to_categorical

train_y = to_categorical(new_label_train)
train_y[:3]

### 1-5. 테스트 데이터 준비

In [ ]:
encoded_test = tokenizer.texts_to_sequences(review_test)
null_index   = [i for i, r in enumerate(encoded_test) if len(r) == 0]
new_review_test = [r for i, r in enumerate(encoded_test) if i not in null_index]
new_label_test  = [l for i, l in enumerate(label_test)  if i not in null_index]

test_X = pad_sequences(new_review_test, maxlen=max_len)
test_y = to_categorical(new_label_test)
print(f'test_X shape: {test_X.shape}')

## 2. 모델 구축 및 컴파일

In [ ]:
from tensorflow.keras.layers import Embedding, LSTM, Dense
from tensorflow.keras.models import Sequential

embedding_dim = 32
lstm_units    = 64
dense_units   = 16
output_units  = 2

model = Sequential([
    Embedding(num_words, embedding_dim, input_length=max_len),
    LSTM(lstm_units),
    Dense(dense_units, activation='tanh'),
    Dense(output_units, activation='softmax')
])
model.summary()

In [ ]:
from tensorflow.keras.optimizers import RMSprop

model.compile(loss='binary_crossentropy', metrics=['accuracy'], optimizer=RMSprop(learning_rate=0.001))

## 3. 모델 학습

In [ ]:
import os
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

os.makedirs('./model', exist_ok=True)
checkpoint_file = './model/best_model_beauty.keras'

es = EarlyStopping(monitor='val_loss', mode='min', patience=3, verbose=1)
mc = ModelCheckpoint(checkpoint_file, monitor='val_loss', save_best_only=True)

history = model.fit(
    train_X, train_y,
    epochs=20, batch_size=128,
    validation_split=0.1,
    callbacks=[es, mc]
)

## 4. 모델 평가

In [ ]:
from tensorflow.keras.models import load_model

model.load_weights(checkpoint_file)
loss, acc = model.evaluate(test_X, test_y)
print(f'손실: {loss:.4f} / 정확도: {acc:.4f}')

In [ ]:
import numpy as np
from sklearn.metrics import classification_report

preds  = model.predict(test_X)
result = [np.argmax(p) for p in preds]

print(classification_report(new_label_test, result, target_names=['부정', '긍정']))

## 5. 예측

In [ ]:
from konlpy.tag import Okt

def analyze_sentiment(text):
    tokens       = Okt().morphs(re.sub('[^ 가-힣]+', ' ', text))
    encoded      = tokenizer.texts_to_sequences([tokens])
    X            = pad_sequences(encoded, maxlen=max_len)
    preds        = model.predict(X, verbose=0)
    labels       = ['부정', '긍정']
    result_index = np.argmax(preds[0])
    return labels[result_index], preds[0][result_index]

# 테스트
reviews = [
    '촉촉하고 발림성도 좋아서 재구매 했어요',
    '피부에 트러블이 생겨서 너무 실망이에요',
    '끈적임이 심하고 냄새도 별로예요',
    '보습력이 뛰어나고 흡수도 빨라요 강추!',
    '자극이 있어서 환불했습니다'
]

for review in reviews:
    result, prob = analyze_sentiment(review)
    print(f'{review[:30]} --> {result} ({prob*100:.1f}%)')

## 6. 배포 (모델 저장)

In [ ]:
import joblib

model.save('./model/sa_model_beauty.keras')
joblib.dump(tokenizer, './model/sa_tokenizer_beauty.pkl')
print('모델 저장 완료')